
## 前処理　データ作成からベクトルDB作成まで

① 記事のHTMLから本文だけを抜き出して保存  
② 本文をチャンク（一定の長さ）に分割  
③ 各チャンクをベクトル化してベクトルDB（Chroma）に保存  




In [ ]:
import requests
from bs4 import BeautifulSoup               #本文だけ抜き出す
from pathlib import Path                    #dataフォルダのファイル一覧を扱う
from dotenv import load_dotenv              #.envからAPIきーをとってくる
from langchain_text_splitters import RecursiveCharacterTextSplitter  #チャンクに分ける
from langchain_google_genai import GoogleGenerativeAIEmbeddings #チャンクをベクトルに変換
from langchain_chroma import Chroma         #ベクトルを保持して検索可能なDB

load_dotenv()

urls = [
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/2070.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/2072.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/2075.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/2090.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/2100.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/2210.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/1350.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/2020.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/1100.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/1130.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/1199.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/2260.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shohi/6101.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shohi/6501.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shohi/6498.htm",
]


# ① 15記事の本文（テキストのみ）を取得して data/ に保存
for url in urls:
    response = requests.get(url)                        #URLのページのHTMLをとってくる（ダウンロード）
    response.encoding = response.apparent_encoding      #文字化け対策
    soup = BeautifulSoup(response.text, "html.parser")  #HTMLを検索できる形（soupオブジェクト）に変換
    body = soup.select_one("div.left-content.contents") #divタグのうち、classが left-content と contents のものを取り出す
    text = body.get_text("\n", strip=True)              #bodyのなかからタグを全部抜いてテキストだけにする　\nはタグの区切りを改行
    code = url.split("/")[-1].replace(".htm", "")       #URLから記事番号を取り出す
    #data/2070.txt というファイルを書き込みモード（"w"）で開いて、本文(text)を書き込む
    with open(f"data/{code}.txt", "w", encoding="utf-8") as f: 
        f.write(text)


#2 チャンクに分割
texts = []
metadatas = []
for path in Path("data").glob("*.txt"):      #Path("data"): 「dataフォルダ」を表すオブジェクトを作る
                                             #.glob("*.txt"): Path("data")の中の.txtのファイルをすべて取り出す
    texts.append(path.read_text(encoding="utf-8"))           # 読み込んだpathの中身をtextsに追加する
    metadatas.append({"source": path.stem})   # path.stem：ファイル(path)の拡張子とフォルダ部分の名前を消す（data/)（文字列じゃないからreplaceは使えない）

splitter =RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)  #分割する道具
chunks = splitter.create_documents(texts, metadatas=metadatas)              #実際に分割　Documentを作る



#3ベクトル化してDBに保存
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
db = Chroma.from_documents(chunks, embeddings, persist_directory="chroma_db") #ベクトル化してchroma_dbフォルダにファイルとして保存


results = db.similarity_search("青色申告特別控除はいくらですか", k=3) #意味がちかいチャンクを３個とってくる

for r in results:
    print(r.metadata)
    print(r.page_content[:100]) #rの先頭100字をとってくる
    print("===" * 20)

## DB読み込み（2回目以降はここから始める）

前処理はもう不要。このセル → 問い合わせセル の順に実行するだけ。
※前処理セルを再実行すると chroma_db に同じチャンクが二重に入るので実行しないこと（作り直すときは chroma_db フォルダを削除してから）

In [ ]:
# 保存済みDBを読み込む（ベクトル化APIを呼ばないので即終了）
from dotenv import load_dotenv
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

load_dotenv()
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")  # 質問文の変換用に必要
db = Chroma(persist_directory="chroma_db", embedding_function=embeddings)       # chroma_dbフォルダから開くだけ

print("チャンク数:", db._collection.count())  # 読み込めたか確認

## 問い合わせ（ベクトル）
① ベクトルDBから似ているチャンクを取ってくる検索器（道具）を作る  
② 質問をベクトルに直し、その道具で各チャンクとの近さを測って近い3件を取ってくる  
③ その3件のチャンクを1つにまとめてLLMに投げる  

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser   #llmの返事から文字列だけとってくる

#部品の準備
retriever = db.as_retriever(search_kwargs={"k":3}) #dbから検索係を作る（上位３件）

prompt = ChatPromptTemplate.from_template(
    "以下の参考情報だけを根拠に質問に答えてください \n"
    "情報がなければ、分かりませんと答えてください \n\n"
    "参考情報: {context}\n\n 質問： {question}"
)

llm = ChatGoogleGenerativeAI(model= "gemini-3.1-flash-lite")

def format_docs(docs): #docsはチャンク3つが入ってくる（documentオブジェクト）
    text = ""
    for d in docs:
        text += d.page_content + "\n\n"   #チャンク3個の本文を、1本の文字列に連結
    return text                             #Documentは page_content と metadata という2つの箱を持つ」と決まってる
                                            # Document
                                            # ├── page_content : 文章そのもの（文字列）
                                            # └── metadata     : 付属情報（辞書）


#実行　質問して、答えと出典を表示
question = "ラーメンの作り方を教えて"
docs = retriever.invoke(question)   #質問と意味が近いチャンクを3個検索
context = format_docs(docs)         # 3個の本文をつなげて1本の文字列に

answer = (prompt | llm | StrOutputParser()).invoke(
    {"context": context, "question": question}
)

sources = sorted({d.metadata["source"] for d in docs}) #3つの出典番号を重複なく集める

print(answer)
if "分かりません" not in answer:
    print("出典：国税庁タックスアンサー"  ,   "、".join("No." + source for source in sources))

## BM25検索器の作成（単語検索）
① 全チャンクを単語に分割し、「どの単語がどのチャンクに何回出るか」を索引にする  
② 質問も単語に分割する  
③ 質問の各単語について、その語を多く含むチャンクほど高得点をつける  
④ このとき、一部のチャンクにしか出てこない語ほど重く数える（「は」「金額」など多くのチャンクに出る語は軽い）  
⑤ 得点の高い順に3件のチャンクを取ってくる  



In [ ]:
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.retrievers import BM25Retriever
from janome.tokenizer import Tokenizer


#チャンクの作成
texts = []
metadatas = []
for path in Path("data").glob("*.txt"):
    texts.append(path.read_text(encoding="utf-8"))
    metadatas.append({"source": path.stem})
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.create_documents(texts, metadatas=metadatas)

#文を単語のリストに切る
tokenizer = Tokenizer()
def tokenize(text):
    return [t.surface for t in tokenizer.tokenize(text)]  #tokenizer.tokenize(text):分を単語ごとに切って、単語の情報を持ったオブジェクトを返す
                                                          #t.surface : オブジェクトから単語そのものをとってくる


# BM25検索器を作る（単語の一致で探す）
bm25_retriever = BM25Retriever.from_documents(chunks,                     # この84個を検索対象に
                                              preprocess_func=tokenize,   # 単語の切り方は、この自作関数で
                                              k=3)


# 動作確認
for d in bm25_retriever.invoke("インボイス制度について教えてください"):
    print(d.metadata, "|", d.page_content[:60])

## ハイブリッド検索器の作成


In [ ]:
from langchain_classic.retrievers import EnsembleRetriever

vector_retriever = db.as_retriever(search_kwargs={"k": 3})  #ベクトル検索

#ハイブリッド検索器
hybrid_retriever = EnsembleRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    weights=[0.5, 0.5],
)

#動作確認
for d in hybrid_retriever.invoke("インボイス制度について教えてください"):
    print(d.metadata, "|", d.page_content[:60])


## 問い合わせ（ハイブリッド）

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

#部品の準備
prompt = ChatPromptTemplate.from_template(
    "以下の参考情報だけを根拠に質問に答えてください \n"
    "情報がなければ、分かりませんと答えてください \n\n"
    "参考情報: {context}\n\n 質問： {question}"
)

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

def format_docs(docs):
    text = ""
    for d in docs:
        text += d.page_content + "\n\n"
    return text


#実行　ハイブリッド検索で答えと出典を表示
question = "インボイス制度について教えてください"

docs = hybrid_retriever.invoke(question)   # ベクトル＋単語一致の両方で検索（最大6件）
context = format_docs(docs)

answer = (prompt | llm | StrOutputParser()).invoke(
    {"context": context, "question": question}
)

sources = sorted({d.metadata["source"] for d in docs})

print(answer)
print("\n出典：国税庁タックスアンサー", "、".join("No." + s for s in sources))


## リランキング
① ハイブリッド検索で、候補チャンクを多めに（各8件）取ってくる  
② その各候補について、質問とチャンクをセットでリランカーに読ませ、関連度スコアを採点する(ベクトルじゃなく、あるニューラルネットワークに質問とチャンクを入れて関連度を出す)  
③ 点数の高い順に上位3件を取ってくる（そしてLLMに投げる）  


In [ ]:
from sentence_transformers import CrossEncoder              # リランカー本体を作るための道具　質問とチャンクを読み比べて関連度を採点するクラス
from langchain_community.retrievers import BM25Retriever    # 単語検索
from langchain_classic.retrievers import EnsembleRetriever  # 複数の検索係をまとめてハイブリッドを作成する


#候補を多めにとるハイブリッド検索器
wide_vector = db.as_retriever(search_kwargs={"k": 8})
wide_bm25 = BM25Retriever.from_documents(chunks, preprocess_func=tokenize, k=8)
wide_hybrid = EnsembleRetriever(retrievers=[wide_vector,wide_bm25], 
                                weights=[0.5, 0.5])


#リランカー
reranker = CrossEncoder("hotchpotch/japanese-reranker-cross-encoder-small-v1")

def rerank(question, docs, top_n=3):
    pairs = [(question, d.page_content) for d in docs]   #「質問」と「各チャンク本文」を1組ずつのリストに
    scores = reranker.predict(pairs)                  #各組の関連度を採点（数値のリストが返る）
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True) #点数が高い順に並び変え
    return [d for d, s in ranked[:top_n]]  #上位top_n個のチャンクを返す


# 動作確認
question = "青色申告特別控除はいくらですか"
candidates = wide_hybrid.invoke(question)   # まず候補を多め（最大16件→重複除いて数件〜十数件）に集める
top_docs = rerank(question, candidates)     # リランカーで採点し直して上位3件に絞る

print("候補数:", len(candidates), "→ 絞り込み:", len(top_docs))
for d in top_docs:
    print(d.metadata["source"], "|", d.page_content[:50].replace("\n", " "))

# 前処理

In [ ]:
# ===== 準備（初回はスクレイピング＋DB作成、2回目以降は読み込みだけ。毎回このセルを実行してOK） =====

import os
import requests
from bs4 import BeautifulSoup               #本文だけ抜き出す
from pathlib import Path                    #dataフォルダのファイル一覧を扱う
from dotenv import load_dotenv              #.envからAPIきーをとってくる
from langchain_text_splitters import RecursiveCharacterTextSplitter  #チャンクに分ける
from langchain_google_genai import GoogleGenerativeAIEmbeddings #チャンクをベクトルに変換
from langchain_chroma import Chroma         #ベクトルを保持して検索可能なDB
from janome.tokenizer import Tokenizer

load_dotenv()
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

urls = [
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/2070.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/2072.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/2075.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/2090.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/2100.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/2210.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/1350.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/2020.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/1100.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/1130.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/1199.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shotoku/2260.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shohi/6101.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shohi/6501.htm",
    "https://www.nta.go.jp/taxes/shiraberu/taxanswer/shohi/6498.htm",
]

# ① 初回だけ：15記事の本文を取得して data/ に保存（chroma_db が無い＝初回）
if not os.path.exists("chroma_db"):
    for url in urls:
        response = requests.get(url)                        #URLのページのHTMLをとってくる（ダウンロード）
        response.encoding = response.apparent_encoding      #文字化け対策
        soup = BeautifulSoup(response.text, "html.parser")  #HTMLを検索できる形（soupオブジェクト）に変換
        body = soup.select_one("div.left-content.contents") #divタグのうち、classが left-content と contents のものを取り出す
        text = body.get_text("\n", strip=True)              #bodyのなかからタグを全部抜いてテキストだけにする　\nはタグの区切りを改行
        code = url.split("/")[-1].replace(".htm", "")       #URLから記事番号を取り出す
        with open(f"data/{code}.txt", "w", encoding="utf-8") as f:
            f.write(text)

# ② 毎回：data/ からチャンクを作る（DB保存にもBM25にも使う）
texts = []
metadatas = []
for path in Path("data").glob("*.txt"):      #Path("data"): 「dataフォルダ」を表すオブジェクトを作る
                                             #.glob("*.txt"): Path("data")の中の.txtのファイルをすべて取り出す
    texts.append(path.read_text(encoding="utf-8"))           # 読み込んだpathの中身をtextsに追加する
    metadatas.append({"source": path.stem})   # path.stem：ファイル(path)の拡張子とフォルダ部分の名前を消す（data/)（文字列じゃないからreplaceは使えない）

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)  #分割する道具
chunks = splitter.create_documents(texts, metadatas=metadatas)              #実際に分割　Documentを作る

# ③ DB：無ければベクトル化して作成、有れば読み込むだけ
if not os.path.exists("chroma_db"):
    db = Chroma.from_documents(chunks, embeddings, persist_directory="chroma_db")  #初回：ベクトル化して保存
    print("新規作成:", db._collection.count())
else:
    db = Chroma(persist_directory="chroma_db", embedding_function=embeddings)       #2回目以降：開くだけ
    print("既存DBを読み込み:", db._collection.count())

# ④ 単語分割の道具（BM25用）
tokenizer = Tokenizer()
def tokenize(text):
    return [t.surface for t in tokenizer.tokenize(text)]  #文を単語ごとに切って、単語だけ取り出す


# ===== 共通パーツ（検索・リランク・生成。以降のセルはこれを使う） =====
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from sentence_transformers import CrossEncoder
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 候補を多めに取るハイブリッド検索器（リランク前提）
wide_vector = db.as_retriever(search_kwargs={"k": 8})
wide_bm25 = BM25Retriever.from_documents(chunks, preprocess_func=tokenize, k=8)
wide_hybrid = EnsembleRetriever(retrievers=[wide_vector, wide_bm25], weights=[0.5, 0.5])

# リランカー（候補を採点し直して上位に絞る。このモデルは512トークンまでなので長い入力は切り詰める）
reranker = CrossEncoder("hotchpotch/japanese-reranker-cross-encoder-small-v1", max_length=512)

def rerank(question, docs, top_n=3):
    pairs = [(question, d.page_content) for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
    return [d for d, s in ranked[:top_n]]

# チャンクを1本の文字列にまとめる
def format_chunk(chunks):
    text = ""
    for chunk in chunks:
        text += chunk.page_content + "\n\n"
    return text

# LLMと基本プロンプト
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
prompt = ChatPromptTemplate.from_template(
    "以下の参考情報だけを根拠に質問に答えてください \n"
    "情報がなければ、分かりませんと答えてください \n\n"
    "参考情報: {context}\n\n 質問: {question}"
)


# CRAG　


### 最初にやること

1. DBから参考情報を取ってくる（ハイブリッド検索＋リランク）
2. その情報で質問に答えられるかを、LLMが3段階で判定する
    - 十分に答えられる
    - 全く関係ない
    - 部分的にしか答えられない

このあと、判定で3つに分かれる。

### 判定が「十分」のとき

- DBの情報だけで回答して終了

### 判定が「関係ない」のとき（DBが使えない＝Webが頼り）

1. Web検索する
2. Web結果で答えられるかを、LLMが判定する
    - 十分 → Webの結果だけで回答して終了
    - 不十分 → 検索語をLLMに言い換えさせて、もう一度Web検索（繰り返す）。上限回数に達したら、今あるWeb結果で回答して終了

### 判定が「部分的」のとき（DBに一部あり＝保険がある）

1. Web検索する
2. Web結果が十分でも不十分でも → DBの情報とWeb結果を合わせて回答して終了
    - Webが不十分でも、言い換え再試行はしない（DBが保険になるため）

### ポイント

- 言い換え再試行が起きるのは「関係ない」のときだけ
- 「部分的」はWebが不十分でも粘らず、DB＋Webで回答する
- どの経路も最後は必ず「回答して終了」に行き着く（無限ループしない）



In [ ]:
# ===== CRAG 3段階 ＋ Web再試行ループ =====
# 検索器・リランク・format_chunk・llm は前処理セルの共通パーツを使う

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.tools import DuckDuckGoSearchRun
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from IPython.display import Image, display


search = DuckDuckGoSearchRun()

MAX_RETRIES = 0  # Web再試行の上限。0 なら再試行なし（＝3段階CRAG）


# --- プロンプト ---
#correct：DBだけで回答
db_answer_prompt = ChatPromptTemplate.from_template(
    "以下の参考情報だけを根拠に質問に答えてください \n"
    "情報がなければ、分かりませんと答えてください \n\n"
    "参考情報: {context}\n\n 質問: {question}"
)

#incorrect：Webだけで回答
web_answer_prompt = ChatPromptTemplate.from_template(
    "以下はWeb検索の結果です。これをもとに質問に分かりやすく答えてください。要約して構いません。 \n\n"
    "Web検索結果: {context}\n\n 質問: {question}"
)

#ambiguous：DBとWebの両方で回答
combined_answer_prompt = ChatPromptTemplate.from_template(
    "以下にはDBの参考情報とWeb検索の結果の両方が含まれています。両方を踏まえて質問に答えてください。 \n\n"
    "参考情報: {context}\n\n 質問: {question}"
)

#DB評価：3段階
grade_prompt = ChatPromptTemplate.from_template(
    "以下の参考情報は、質問に答えるのにどれくらい役立ちますか？次の3つから1語だけ英語で答えてください。 \n"
    "・十分に答えられる → correct \n"
    "・全く関係ない → incorrect \n"
    "・部分的にしか答えられない → ambiguous\n\n"
    "参考情報: {context}\n\n 質問: {question}"
)

#Web評価：十分か
web_grade_prompt = ChatPromptTemplate.from_template(
    "以下はWeb検索の結果です。質問に答えるのに十分ですか？"
    "十分なら「はい」、不十分なら「いいえ」とだけ答えてください。 \n\n"
    "Web検索結果: {context}\n\n 質問: {question}"
)

#検索語の言い換え
rewrite_prompt = ChatPromptTemplate.from_template(
    "次の質問を、Web検索でヒットしやすい検索キーワードに言い換えてください。"
    "言い換えた検索語だけを出力してください。 \n"
    "質問: {question}"
)


# --- State ---
class State(TypedDict):
    question:     str   # 元の質問（変えない）
    search_query: str   # Web検索用クエリ（言い換えで変わる）
    context:      str   # DBの参考情報
    web_context:  str   # Web検索の結果
    sources:      list  # 出典
    grade:        str   # DBの3段階評価
    web_ok:       str   # Web結果が十分か（yes / no）
    attempts:     int   # 再試行回数
    answer:       str   # 最終回答


# --- ノード ---

# DB検索＋リランク → context と sources を埋める
def retrieve_node(state):
    candidates = wide_hybrid.invoke(state["question"])
    top_chunks = rerank(state["question"], candidates)
    return {"context": format_chunk(top_chunks),
            "sources": sorted({chunk.metadata["source"] for chunk in top_chunks})}


# DBの3段階評価 → grade を埋める
def grade_node(state):
    result = (grade_prompt | llm | StrOutputParser()).invoke(
        {"context": state["context"], "question": state["question"]}
    ).lower()                      # 表記ゆれ対策で小文字化

    if "incorrect" in result:
        grade = "incorrect"
    elif "ambiguous" in result:
        grade = "ambiguous"
    elif "correct" in result:
        grade = "correct"
    else:
        grade = "ambiguous"        # 判定不能は安全側（Web確認）に倒す

    return {"grade": grade}


# Web検索：search_query（無ければ元の質問）で検索 → web_context を埋める
def web_search_node(state):
    query = state.get("search_query") or state["question"]
    web = search.invoke(query)
    return {"web_context": web}


# Web結果が十分か評価 → web_ok を埋める
def grade_web_node(state):
    result = (web_grade_prompt | llm | StrOutputParser()).invoke(
        {"context": state["web_context"], "question": state["question"]}
    )
    return {"web_ok": "yes" if "はい" in result else "no"}


# 検索語を言い換える → search_query を更新、attempts を +1
def rewrite_node(state):
    new_query = (rewrite_prompt | llm | StrOutputParser()).invoke(
        {"question": state["question"]}
    )
    return {"search_query": new_query.strip(),
            "attempts": state["attempts"] + 1}


# 回答：grade に応じて「使う文脈・出典・プロンプト」を切り替える
def generate_answer_node(state):
    if state["grade"] == "correct":
        context = state["context"]
        sources = state["sources"]
        used_prompt = db_answer_prompt

    elif state["grade"] == "incorrect":
        context = state["web_context"]
        sources = ["Web検索"]
        used_prompt = web_answer_prompt

    else:  # ambiguous：DB＋Web
        context = state["context"] + "\n\n【Web検索】\n" + state["web_context"]
        sources = state["sources"] + ["Web検索"]
        used_prompt = combined_answer_prompt

    answer = (used_prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": state["question"]}
    )
    return {"answer": answer, "sources": sources}


# --- 分岐 ---

# DB評価のあと：correct→回答、それ以外→Web検索
def route_after_grade(state):
    if state["grade"] == "correct":
        return "generate_answer"
    else:
        return "web_search"


# Web評価のあと：十分/上限→回答、不十分&再試行可→言い換え
def route_after_web(state):
    if state["web_ok"] == "yes":
        return "generate_answer"

    # Web不十分のとき
    if state["grade"] == "incorrect" and state["attempts"] < MAX_RETRIES:
        return "rewrite"          # incorrect：Webが頼り → 検索語を変えて再検索

    return "generate_answer"      # ambiguous：DBが保険 / 上限 → いまの結果で回答



# --- グラフ ---
builder = StateGraph(State)
builder.add_node("retrieve", retrieve_node)
builder.add_node("grade", grade_node)
builder.add_node("web_search", web_search_node)
builder.add_node("grade_web", grade_web_node)
builder.add_node("rewrite", rewrite_node)
builder.add_node("generate_answer", generate_answer_node)


builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "grade")

builder.add_conditional_edges("grade",                               #第一引数： どのノードの直後に分岐させるか
                              route_after_grade,                     #第二引数： 判定関数
                              {"generate_answer": "generate_answer", #第三引数： 判定関数が返した文字列を実際のノード名に変換する表  
                               "web_search":      "web_search"})

builder.add_edge("web_search", "grade_web")

builder.add_conditional_edges("grade_web",
                              route_after_web,
                              {"generate_answer": "generate_answer",
                               "rewrite":         "rewrite"})

builder.add_edge("rewrite", "web_search")
builder.add_edge("generate_answer", END)

graph = builder.compile()


# --- 実行 ---
for question in ["ふるさと納税の控除上限額の計算方法を教えて"]:
    result = graph.invoke({"question": question, "attempts": 0})
    print("質問:", question)
    print(result["answer"])
    print("出典:", "、".join(result["sources"]))
    print("-" * 40)

display(Image(graph.get_graph().draw_mermaid_png()))


## AgenticRAG

① 質問を渡す  
② LLMが「次に何をするか」を自分で決める  
     ・「まずDBを調べよう」→ db_search を呼ぶ  
③ ツールの結果を見て、LLMがまた考える  
     ・「これで十分に答えられる」→ 回答して終了  
     ・「まだ足りない」→ web_search を呼ぶ  
④ ②③を、LLMが「もう答えられる」と判断するまで繰り返す


In [ ]:
# ===== 完全自律型 Agentic RAG（ReAct：LLMがツールを自分で選ぶ）=====

from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from sentence_transformers import CrossEncoder
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langchain.agents import create_agent          # 新しい書き方（旧 create_react_agent）
from IPython.display import Image, display


# --------------------------------------------------
# 1. 検索の部品（DB検索＋リランク）
# --------------------------------------------------
wide_vector = db.as_retriever(search_kwargs={"k": 8})
wide_bm25 = BM25Retriever.from_documents(chunks, preprocess_func=tokenize, k=8)
wide_hybrid = EnsembleRetriever(retrievers=[wide_vector, wide_bm25], weights=[0.5, 0.5])

reranker = CrossEncoder("hotchpotch/japanese-reranker-cross-encoder-small-v1")


def rerank(question, chunks, top_n=3):
    pairs = [(question, chunk.page_content) for chunk in chunks]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(chunks, scores), key=lambda x: x[1], reverse=True)
    return [chunk for chunk, score in ranked[:top_n]]


def format_chunk(chunks):
    text = ""
    for chunk in chunks:
        text += chunk.page_content + "\n\n"
    return text


# --------------------------------------------------
# 2. LLM と Web検索
# --------------------------------------------------
llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
search = DuckDuckGoSearchRun()


# --------------------------------------------------
# 3. ツール（LLMが自分で選んで使う道具）
# --------------------------------------------------
@tool
def db_search(query: str) -> str:
    """国税庁タックスアンサーのDBから税務の参考情報を検索する。
    日本の税金・控除・申告など、制度に関する質問にはまずこれを使う。"""
    candidates = wide_hybrid.invoke(query)
    top_chunks = rerank(query, candidates)
    return format_chunk(top_chunks)


@tool
def web_search(query: str) -> str:
    """最新情報や、DBに載っていない一般的な情報をWebから検索する。
    DBで十分な答えが得られなかったときに使う。"""
    return search.invoke(query)


# --------------------------------------------------
# 4. エージェント（ツールを渡すだけ。流れの判断はLLMが行う）
# --------------------------------------------------
system_prompt = (
    "あなたは日本の税務に詳しいアシスタントです。"
    "税務の質問にはまず db_search を使ってDBを調べてください。"
    "DBで十分に答えられないと判断したときだけ web_search を使ってください。"
    "十分な情報が集まったら、それを根拠に日本語で分かりやすく答えてください。"
)

agent = create_agent(model=llm, 
                     tools=[db_search, web_search], 
                     system_prompt=system_prompt
                     )


# --------------------------------------------------
# 5. 実行
# --------------------------------------------------
question = "基礎控除はいくらですか？"   # DBに入っている話題で試す

result = agent.invoke({"messages": [{"role": "user", "content": question}]})




# --------------------------------------------------
# 6. 図
# --------------------------------------------------
display(Image(agent.get_graph().draw_mermaid_png()))



## セマンティック分割

In [ ]:
# ===== セマンティック分割（意味の切れ目でチャンク化）＋ リランク検索 =====
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_experimental.text_splitter import SemanticChunker
from langchain_chroma import Chroma
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.output_parsers import StrOutputParser


# --- 1. 意味の切れ目で分割（切れ目の判定だけローカル埋め込み。検索には使わない） ---
split_embeddings = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-small")

splitter = SemanticChunker(
    split_embeddings,
    sentence_split_regex=r"(?<=[。！？])|\n",   # 日本語の文末と改行で文に区切る（既定は英語用で効かない）
    breakpoint_threshold_type="percentile",     # 距離が上位（既定95%）に入る境目で切る
    min_chunk_size=150,                          # 小さすぎるチャンク（見出し・表）を隣と結合
)
semantic_chunks = splitter.create_documents(texts, metadatas=metadatas)
print("セマンティック分割:", len(semantic_chunks), "チャンク")


# --- 2. セマンティック用DB（検索の埋め込みは固定長DBと同じGeminiで揃える） ---
db_semantic = Chroma.from_documents(semantic_chunks, embeddings, persist_directory="chroma_db_semantic")


# --- 3. 検索器（固定長のリランク版と同じ構成を、セマンティックDBに対して） ---
sem_vector = db_semantic.as_retriever(search_kwargs={"k": 8})
sem_bm25 = BM25Retriever.from_documents(semantic_chunks, preprocess_func=tokenize, k=8)
sem_hybrid = EnsembleRetriever(retrievers=[sem_vector, sem_bm25], weights=[0.5, 0.5])


# --- 4. 問い合わせ（候補を多め → リランクで上位3件 → LLM） ---
question = "青色申告特別控除はいくらですか"

candidates = sem_hybrid.invoke(question)
top_docs = rerank(question, candidates)
context = format_chunk(top_docs)

answer = (prompt | llm | StrOutputParser()).invoke(
    {"context": context, "question": question}
)
sources = sorted({d.metadata["source"] for d in top_docs})

print(answer)
print("出典:", "、".join("No." + s for s in sources))


## VLM

In [ ]:
# ===== VLM ＋ RAG（画像を読み取り、税務DBで補強して回答）=====
import base64
from langchain_core.messages import HumanMessage

# 0. 画像ファイルを「Geminiに送れる文字列」に変換する
image_path = "sample.png"                         # 質問したい画像のパス
with open(image_path, "rb") as f:                 # 画像をバイナリ(bytes)で開く
    image_data = base64.b64encode(f.read()).decode("utf-8")  # bytes(バイト列)→base64(文字列)→str に変換


image_block = {"type": "image_url",                             
               "image_url": {"url": f"data:image/png;base64,{image_data}"}}  # PNGをbase64で直接埋め込み

question = "この源泉徴収票をもとに、私が受けられる控除について教えて"   # 最終的に答えたい質問


# ① 画像を見て「DB検索に使うキーワード」をGeminiに作らせる
query_msg = HumanMessage(content=[               
    {"type": "text",
     "text": f"次の質問に答えるため、この画像から税務DBを検索するのに適したキーワードを1行で出してください。\n質問: {question}"},
    image_block,
])
search_query = llm.invoke([query_msg]).content   # Geminiに渡す→返事(AIメッセージ)の.contentで本文テキストだけ取り出す
print("検索クエリ:", search_query)                # 実際にどんなキーワードが作られたか確認


# ② そのキーワードで税務DBを検索（今までのRAGと同じ部品を再利用）
candidates = wide_hybrid.invoke(search_query)    # ベクトル＋単語で候補を多めに集める
top_chunks = rerank(search_query, candidates)    # リランカーで上位3件に絞る
context = format_chunk(top_chunks)               # 3件の本文を1本の文字列にまとめる


# ③ 画像＋②で得た記事＋質問 をGeminiに渡して最終回答
answer_msg = HumanMessage(content=[              # また「テキスト＋画像」の1通を作る
    # 参考情報(context)と質問を埋め込んだ指示
    {"type": "text",
     "text": f"画像の内容と、以下の参考情報（税務DB）を踏まえて質問に答えてください。\n\n参考情報:\n{context}\n\n質問: {question}"},
    image_block,                                 # 画像も一緒に渡す
])
answer = llm.invoke([answer_msg]).content        # Geminiに渡して最終回答のテキストを受け取る

print(answer)
# 使った記事の番号（出典）を重複なく並べて表示
print("出典:", "、".join("No." + s for s in sorted({c.metadata["source"] for c in top_chunks})))


## GraphRAG

まず記事をllmが読んで、ノードをエッジを作成してグラフを作成してくれる。

次にノードにベクトルを寄与してあげる。
そして質問もベクトルにしてあげて、各ノードと質問のコサイン類似度を見て、類似度が高いノードを決める。

そしてそのノードから今回は2個分ノードまでの情報（どのノードがどう繋がっているか）をllｍに渡して、自然な文章で質問に答えさせる

In [ ]:
# ===== GraphRAG（知識グラフ検索）※Neo4j不使用・networkxで最小構成 =====
import os, pickle
import networkx as nx
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

GRAPH_CACHE = "graph.pickle"

# --- 1. 知識グラフを作る（初回だけ。関連する数記事からエンティティ・関係を抽出）---
if os.path.exists(GRAPH_CACHE):
    with open(GRAPH_CACHE, "rb") as f:
        G = pickle.load(f)                                # 2回目以降は読み込むだけ
else:
    # 関係が濃い「事業・青色申告」系だけに絞る（少ない・つながりやすい）
    TARGET = {"2070", "2072", "2075", "2090", "2100", "2210"}
    docs = [Document(page_content=t, metadata={"source": m["source"]})
            for t, m in zip(texts, metadatas) if m["source"] in TARGET]  # 6本だけ
    transformer = LLMGraphTransformer(llm=llm)            # Geminiでエンティティ・関係を抽出する道具
    graph_docs = transformer.convert_to_graph_documents(docs)   # ← ここで記事6本ぶんLLMを呼んで、エンティティ（ノード候補）、関係（エッジ候補）を取得

    G = nx.DiGraph()                                      # 空の入れ物（グラフ）を用意
    for gd in graph_docs:
        for node in gd.nodes:
            G.add_node(node.id, type=node.type)          # エンティティ＝ノードを登録
        for rel in gd.relationships:
            G.add_edge(rel.source.id, rel.target.id, type=rel.type)  # 関係＝エッジを登録

    with open(GRAPH_CACHE, "wb") as f:
        pickle.dump(G, f)                                 # 完成したグラフを保存して次回から再利用

print("ノード数:", G.number_of_nodes(), "／ エッジ数:", G.number_of_edges())


# --- ①エンティティリンキング用：全ノード名をベクトルDB化（1回だけ・ローカル埋め込み）---
node_emb = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-small")
node_store = Chroma.from_texts(list(G.nodes), embedding=node_emb,
                               collection_metadata={"hnsw:space": "cosine"})  # 類似度はコサインで測る


# --- 2. グラフ検索：質問に意味が近いノードを起点に、2ホップ分の関係を集める ---
def graph_context(question, top_k=3, min_sim=0.8):
    # ① 質問に意味が近いノードを類似度つきで上位top_k件取得
    hits = node_store.similarity_search_with_relevance_scores(question, k=top_k)
    matched = [doc.page_content for doc, score in hits if score >= min_sim]  # 類似度min_sim以上だけ採用

    # ③ マッチした各ノードから半径2の部分グラフを取り出し、その全エッジを集める
    lines, seen = [], set()
    for n in matched:
        sub = nx.ego_graph(G, n, radius=2, undirected=True)  # nから2ホップ以内で繋がるノード＆エッジ
        for src, tgt, d in sub.edges(data=True):          # 部分グラフの全エッジ（始点, 終点, 情報）
            triple = f"{src} -[{d['type']}]-> {tgt}"
            if triple not in seen:                        # 起点が複数だと重複するので除く
                seen.add(triple)
                lines.append(triple)
    return "\n".join(lines)                               # 青色申告特別控除 -[REQUIRES]-> 複式簿記 のような関係を並べたテキスト


# --- 3. 問い合わせ（グラフの関係を文脈にして回答）---
question = "青色申告特別控除の要件は何ですか"

context = graph_context(question)
print("グラフから取得した関係:\n", context, "\n")

answer = (prompt | llm | StrOutputParser()).invoke(
    {"context": context, "question": question}
)
print(answer)


In [ ]:
# ===== グラフの可視化（注目エンティティの周辺だけ・pyvis）=====
from pyvis.network import Network

center = "青色申告特別控除"                 # 見たいエンティティ（変えられる）
nodes = {center} | set(G.successors(center)) | set(G.predecessors(center))  # 中心＋隣接だけ
sub = G.subgraph(nodes)                     # 部分グラフ

net = Network(height="600px", width="100%", directed=True, notebook=True, cdn_resources="in_line")
for n in sub.nodes:
    net.add_node(n, label=n)
for s, t, d in sub.edges(data=True):
    net.add_edge(s, t, label=d.get("type", ""))   # 関係名をエッジに表示

net.save_graph("graph.html")                # graph.html を作る


## Neo4jを使って作成

In [ ]:
# ===== GraphRAG（Neo4j版）※Aura に保存し Cypher で多ホップ検索 =====
from langchain_neo4j import Neo4jGraph, Neo4jVector
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_huggingface import HuggingFaceEmbeddings

graph = Neo4jGraph()   # .env の NEO4J_URI / NEO4J_USERNAME / NEO4J_PASSWORD を自動で読む

# --- 1. グラフをNeo4jに保存（空のときだけ実行）---
if graph.query("MATCH (n) RETURN count(n) AS c")[0]["c"] == 0:   #ノードが何個あるのか聞く　0なら中を実行
    docs = [Document(page_content=t, metadata={"source": m["source"]})
            for t, m in zip(texts, metadatas)]  # 15本すべてDocument型にする（LangChain共通でこれしか使えないから）　Documentはpage_contentとmetadataを持つ（metadataの中身は自由）
    graph_docs = LLMGraphTransformer(llm=llm).convert_to_graph_documents(docs)  # 記事15本ぶんLLMを呼んで、エンティティ（ノード候補）、関係（エッジ候補）を取得
    graph.add_graph_documents(graph_docs, baseEntityLabel=True, include_source=False)  # Auraへ書き込み
    # baseEntityLabel=True → 全ノードに __Entity__ ラベルが付き、次のベクトル索引の対象にできる

# --- ①エンティティリンキング用：各ノードにembedding（ベクトル）を付けてベクトル検索できるようにする
node_emb = HuggingFaceEmbeddings(model_name="intfloat/multilingual-e5-small")
vector_index = Neo4jVector.from_existing_graph(     #「既にNeo4jにあるノードを、意味で検索できるようにする」関数
    embedding=node_emb,
    node_label="__Entity__",              # 対象＝Entityの目印が付いた全ノード
    text_node_properties=["id"],          # ノード名(id)をベクトル化する
    embedding_node_property="embedding",  # 作ったベクトルをこのプロパティに保存
    retrieval_query="RETURN node.id AS text, score, {} AS metadata",  # 検索結果はノード名だけ返す
)

# プロパティ
# 青色申告特別控除 ノード
# ├─ id : "青色申告特別控除"
# └─ embedding : [0.12, -0.03, ...]   ← 新しく追加されたベクトル


# 【Neo4jのノード】プロパティ        【Python側】d（Document）の中身
# ├─ id        : "青色申告特別控除"   ├─ page_content : "青色申告特別控除"
# └─ embedding : [0.12, ...]         └─ metadata     : {}


# --- 2. グラフ検索：質問に近いノードを起点に、Cypherで2ホップ分の関係を集める ---
def graph_context(question, k=3):
    ents = [d.page_content for d in vector_index.similarity_search(question, k=k)]  # 質問に近いノード名　　similarity_searchはdocumentを返す　→.page_contentが使える　d.page_contentは質問と意味が近いノード名
    rows = graph.query(
        """
        MATCH (n)-[r*1..2]-(m)          // nから2ホップ以内のパス（矢印の向きは無視）
        WHERE n.id IN $ents
        UNWIND r AS rel                  // パス上の関係を1本ずつに展開
        RETURN DISTINCT startNode(rel).id AS src, type(rel) AS rel, endNode(rel).id AS tgt
        """,
        params={"ents": ents},
    )
    return "\n".join(f"{row['src']} -[{row['rel']}]-> {row['tgt']}" for row in rows)


# --- 3. 問い合わせ（グラフの関係を文脈にして回答）---
question = "青色申告特別控除の要件は何ですか"

context = graph_context(question)
print("グラフから取得した関係:\n", context, "\n")

answer = (prompt | llm | StrOutputParser()).invoke(
    {"context": context, "question": question}
)
print(answer)


## 文書検索 + GraphRAGのハイブリッド

In [ ]:
# ===== 文書検索（ハイブリッド＋リランク）× GraphRAG を組み合わせて回答 =====
from langchain_core.output_parsers import StrOutputParser

question = "青色申告特別控除の要件は何ですか"

# ① 文書検索：ハイブリッド → リランクで上位3件（本文を取ってくる）
candidates = wide_hybrid.invoke(question)
top_chunks = rerank(question, candidates)
doc_context = format_chunk(top_chunks)

# ② グラフ検索：質問に近いノードから2ホップ分の関係（Neo4j版の graph_context）
graph_ctx = graph_context(question)

# ③ 文書（本文）とグラフ（関係）を1つの文脈にまとめる
context = (
    "【関連する文書】\n" + doc_context +
    "\n【エンティティの関係】\n" + graph_ctx
)

# ④ 回答（本文＋関係の両方を根拠にLLMが答える）
answer = (prompt | llm | StrOutputParser()).invoke(
    {"context": context, "question": question}
)
print(answer)
print("出典:", "、".join("No." + s for s in sorted({c.metadata["source"] for c in top_chunks})))
